# Knocking out an induction head

[*Induction heads in the wild*](https://barmag.github.io/) found the hand-crafted circuit inside GPT-2 small and Pythia-160m: a previous-token head (GPT-2 **L4H11**, Pythia **L3H2**) feeding a K-composed induction head (GPT-2 **L5H5**, Pythia **L4H6**), with a copying OV. Every one of those findings was **correlational**. A matrix looked right. A score was high. A rank was 1.

That notebook ended on a promise:

> Knock out L4H11 and L5H5, or L3H2 and L4H6, and watch whether the induction score collapses. That is the next notebook.

This is that notebook. We cut heads out of the forward pass and watch what breaks.

**Hypotheses.**

1. **H1 (knock-out).** If L5H5 / L4H6 *is* the induction head, removing it should collapse induction behavior — the model should lose its ability to predict the repeated half of a repeated random sequence. Predicted wrinkle: only a *partial* collapse, because the wild notebook already showed the behavior lives in a small cluster (GPT-2's L6H9 scores 0.917 against L5H5's 0.930).
2. **H2 (mediation).** If K-composition is causal, removing the *previous-token* head should kill the *induction* head's attention stripe — an effect one layer removed from the intervention.
3. **H3 (stretch).** If Pythia L4H6's negative OV eigenvalue mass is non-copying work sharing the same head, surgically removing those directions should preserve induction while costing something elsewhere.

**Papers.** Olsson et al. 2022, [*In-context Learning and Induction Heads*](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) (ablations, per-token induction loss); Wang et al. 2023, [*Interpretability in the Wild*](https://arxiv.org/abs/2211.00593) (backup heads — the hydra effect); Elhage et al. 2021, [*A Mathematical Framework for Transformer Circuits*](https://transformer-circuits.pub/2021/framework/index.html) (OV eigenvalues).


In [1]:
import os
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "11.0.0")  # Strix Halo gfx1151 runs the gfx1100 wheels

import subprocess
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch
from transformer_lens import HookedTransformer, utils

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"device={device}  seed={SEED}  commit={commit}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

device=cuda  seed=42  commit=9889c69
GPU: AMD Radeon 8060S Graphics


## Act 0 — The yardstick

Olsson et al. measure induction with a **repeated random sequence**: sample a block of random tokens, concatenate it with itself, and ask the model to predict. The first copy is unpredictable by construction — random tokens carry no signal, so the model can do no better than its prior. The second copy is perfectly predictable *if and only if* the model can look back, find where the current token appeared before, and copy what followed.

So the difference between those two halves is the behavior itself, measured in nats:

> **induction loss gap** = mean loss on the first (unrepeated) half − mean loss on the second (repeated) half.

Positive by construction when induction works. Every intervention in this notebook is scored by how much of this gap it destroys:

> **collapse fraction** = (gap_clean − gap_ablated) / gap_clean

0% means the intervention did nothing. 100% means induction is gone.

Both models, same seed and batch as the previous notebook, so the numbers line up across the two.

In [2]:
gpt2 = HookedTransformer.from_pretrained("gpt2", device=device)
pythia = HookedTransformer.from_pretrained("pythia-160m", device=device)
MODELS = {"gpt2": gpt2, "pythia": pythia}

BATCH, T, GATE = 32, 50, 0.2


def repeated_tokens(model, batch=BATCH, block=T, seed=SEED):
    """[BOS, block, block] -> [batch, 2T+1]. Same generator and seed as the previous notebook."""
    g = torch.Generator().manual_seed(seed)
    block_toks = torch.randint(100, 50_000, (batch, block), generator=g)  # both vocabs exceed 50k
    bos = torch.full((batch, 1), model.tokenizer.bos_token_id, dtype=torch.long)
    return torch.cat([bos, block_toks, block_toks], dim=1).to(model.cfg.device)


def halves(lpt):
    """loss_per_token[:, i] scores predicting token i+1. Cols 0..T-1 = first copy, T..2T-1 = second."""
    return lpt[:, 0:T].mean().item(), lpt[:, T:2 * T].mean().item()


def loss_gap(lpt):
    first, second = halves(lpt)
    return first - second


TOKENS, BASE = {}, {}
for name, model in MODELS.items():
    TOKENS[name] = repeated_tokens(model)
    lpt = model(TOKENS[name], return_type="loss", loss_per_token=True)
    first, second = halves(lpt)
    BASE[name] = {"first": first, "second": second, "gap": first - second}
    print(f"{name}: first(unrepeated)={first:6.3f}  second(repeated)={second:5.3f}  "
          f"gap={first - second:6.3f} nats")


def collapse(gap_abl, name):
    return (BASE[name]["gap"] - gap_abl) / BASE[name]["gap"]

`torch_dtype` is deprecated! Use `dtype` instead!


/home/yassermakram/code/fanous-llm-lens/.venv/lib64/python3.11/site-packages/torch/nn/modules/module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at ../c10/hip/HIPAllocatorConfig.h:29.)
  return t.to(


Loaded pretrained model gpt2 into HookedTransformer


Loaded pretrained model pythia-160m into HookedTransformer


/home/yassermakram/code/fanous-llm-lens/.venv/lib64/python3.11/site-packages/transformer_lens/utilities/attention.py:27: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, w, b_).reshape(input.shape[0], input.shape[1], b.shape[0], b.shape[1])


gpt2: first(unrepeated)=13.190  second(repeated)=0.481  gap=12.709 nats


pythia: first(unrepeated)=23.443  second(repeated)=5.561  gap=17.881 nats


### Results — the gap is real, and the two models are not alike

**GPT-2 small:** 13.190 nats on the unrepeated half, **0.481** on the repeated half — a gap of **12.709 nats**. The second time it sees the sequence, GPT-2 predicts it at about half a nat per token. That is very close to knowing the answer.

**Pythia-160m:** 23.443 → **5.561**, a gap of **17.881 nats**. The gap is *larger*, but read the second number rather than the difference: Pythia still pays 5.6 nats per token on text it has already seen once. It is copying, but nothing like as cleanly as GPT-2 — despite Pythia's induction head scoring *higher* on the attention diagnostic in the previous notebook (0.987 vs 0.930). Attending to the right place and writing the right token are two different jobs, and the previous notebook already saw the seam: Pythia's copying-OV score was 0.423 against GPT-2's 0.816.

Both first-half losses sit above ln(50000) ≈ 10.8 nats, the cost of guessing uniformly among the token IDs we sampled from. That is expected: a language model is not a uniform prior over token IDs. It finds a random draw from the middle of the vocabulary *less* likely than chance would, and Pythia — whose tokenizer and training mix differ from GPT-2's — finds it much less likely still. This is why the two models' absolute nats are not comparable to each other, and why every intervention below is scored as a **fraction of that model's own gap** rather than in raw nats.

The gap is the yardstick. Now we start cutting.